<a href="https://colab.research.google.com/github/MaJiMe0123/for-hello-world-/blob/main/Fuzzy_Theory_Based_Tip_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install scikit-fuzzy gradio

import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt
import gradio as gr

# 퍼지 로직 엔진
class FuzzyEngine:
    def __init__(self):
        # 기본 설정값
        self.params = {
            'svc': {'Poor': [0, 0, 5], 'Good': [2, 5, 8], 'Excellent': [5, 10, 10]},
            'fd': {'Rancid': [0, 0, 6], 'Delicious': [4, 10, 10]},
            'tip': {'Cheap': [0, 5, 10], 'Average': [10, 15, 20], 'Generous': [20, 25, 30]}
        }
        self.rules_matrix = {
            ('Poor', 'Rancid'): 'Cheap', ('Poor', 'Delicious'): 'Average',
            ('Good', 'Rancid'): 'Cheap', ('Good', 'Delicious'): 'Average',
            ('Excellent', 'Rancid'): 'Average', ('Excellent', 'Delicious'): 'Generous'
        }

    def compute(self, s_v, f_v):
        x_svc = np.arange(0, 10.1, 0.1)
        x_fd = np.arange(0, 10.1, 0.1)
        x_tip = np.arange(0, 30.1, 0.1)

        # 멤버십
        mfs = {d: {n: fuzz.trimf(np.arange(0, 10.1 if d != 'tip' else 30.1, 0.1), v)
               for n, v in s.items()} for d, s in self.params.items()}

        # 퍼지화
        svc_lv = {n: fuzz.interp_membership(x_svc, mfs['svc'][n], s_v) for n in self.params['svc'].keys()}
        fd_lv = {n: fuzz.interp_membership(x_fd, mfs['fd'][n], f_v) for n in self.params['fd'].keys()}

        # 추론(Mamdani)
        active_tips = []
        for (s_label, f_label), t_label in self.rules_matrix.items():
            weight = np.fmin(svc_lv[s_label], fd_lv[f_label])
            active_tips.append(np.fmin(weight, mfs['tip'][t_label]))

        aggregated = np.fmax.reduce(active_tips)
        res = fuzz.defuzz(x_tip, aggregated, 'centroid') if np.sum(aggregated) > 0 else 0

        # 시각화
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        plt.style.use('dark_background')
        fig.patch.set_facecolor('#1E1E2E')

        colors = ['#F38BA8', '#A6E3A1', '#89B4FA']
        # Service
        for i, (n, v) in enumerate(mfs['svc'].items()): axes[0,0].plot(x_svc, v, color=colors[i], label=n)
        axes[0,0].axvline(s_v, color='white', ls='--', alpha=0.5)
        axes[0,0].set_title("SERVICE ANALYSIS")

        # Food
        for i, (n, v) in enumerate(mfs['fd'].items()): axes[1,0].plot(x_fd, v, color=colors[i], label=n)
        axes[1,0].axvline(f_v, color='white', ls='--', alpha=0.5)
        axes[1,0].set_title("FOOD ANALYSIS")

        # Activation
        for act in active_tips: axes[0,1].fill_between(x_tip, 0, act, color='#89B4FA', alpha=0.2)
        axes[0,1].set_title("RULE ACTIVATION")

        # Result
        axes[1,1].fill_between(x_tip, 0, aggregated, color='#89B4FA', alpha=0.4)
        axes[1,1].plot(x_tip, aggregated, color='#89B4FA', lw=2)
        axes[1,1].axvline(res, color='#A6E3A1', lw=4)
        axes[1,1].set_title(f"FINAL TIP VALUE: {res:.2f}%")

        plt.tight_layout()
        return fig, f"최종 추론 결과: {res:.2f}%"

engine = FuzzyEngine()

# Gradio 웹 인터페이스 설정
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 FUZZY MASTER - Colab Edition")
    gr.Markdown("### Made by Master")

    with gr.Row():
        with gr.Column():
            s_slider = gr.Slider(0, 10, value=5, label="Service Quality")
            f_slider = gr.Slider(0, 10, value=5, label="Food Quality")
            run_btn = gr.Button("RUN INFERENCE", variant="primary")

        with gr.Column():
            result_text = gr.Textbox(label="Result Value")
            result_plot = gr.Plot(label="Fuzzy Analysis Visualization")

    run_btn.click(fn=engine.compute, inputs=[s_slider, f_slider], outputs=[result_plot, result_text])

demo.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 14.9 MB/s eta 0:00:00


/tmp/ipykernel_5627/2358842144.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cd420feb5bf7c02831.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
